# E01: 
train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

# Solution:

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
words = open("names.txt", 'r').read().splitlines()

In [3]:
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [4]:
len(words)

32033

In [5]:
min(words, key=len), len(min(words, key=len))

('an', 2)

In [6]:
max(words, key=len), len(max(words, key=len))

('muhammadibrahim', 15)

<a id="Final Solution:"></a>



In [7]:
# Get list of all the characters
chars = ['.'] + sorted(list(set(''.join(words)))) # ., a, b, c, d, .... x, y, z
len(chars)

27

In [22]:
# Create dictionary for mapping single character to its ID
stoi = {s:i for i, s in enumerate(chars)}

# Create another dictionary for reverse mapping
itos = {i:s for s, i in stoi.items()}

# Check if its correct
stoi['.'], stoi['a'], itos[0]

(0, 1, '.')

In [31]:
# Get all 27*27 combinations (26 alphabets and one '.')
all_combinations = [a + b for a in chars for b in chars]

# Create dictionary for mapping combination to its ID
ctoi={c:i for i,c in enumerate(all_combinations)}

# Create another dictionary for reverse mapping
itoc={i:c for c, i in ctoi.items()}

# Check if its correct
len(ctoi), ctoi['..'], itoc[1]

(729, 0, '.a')

**Note:** A universal dictionary (`combinations + chars`) would not work in this case because the character IDs would fall in the range **729–755**, while the output classes are expected to be in the range **0–26**.

Doing it the other way around (`chars + combinations`) would not work either, because the combination IDs would fall in the range **27–755**, while `one_hot` for the input expects IDs in the range **0–728** when `num_classes=729`.

Therefore, separate dictionaries are needed:
- `ctoi` → character → IDs `0–26`
- `stoi` → combination → IDs `0–728`


In [ ]:
# Creating the dataset
xs, ys = [], []
for w in words:
    chs = ['.','.'] + list(w) + ['.']  # eg ['.', '.', 'e', 'm', 'm', 'a', '.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ch12 = ch1+ch2
        idx12 = ctoi[ch12]
        idx3 = stoi[ch3]
        xs.append(idx12)
        ys.append(idx3)

# Converting dataset into tensor
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.numel()
xs, ys, num 

(tensor([  0,   5, 148,  ..., 727, 701, 726]),
 tensor([ 5, 13, 13,  ..., 26, 24,  0]),
 228146)

In [11]:
# Setting up the generator and initializing weights with random values
g = torch.Generator().manual_seed(312312)
W = torch.randn((729, 27), generator=g, requires_grad=True)

In [ ]:
# Training loop
for i in range(50):
    #----------Forward pass------------
    xenc = F.one_hot(xs, num_classes=len(all_combinations)).float() #729
    logits = xenc @ W # (228146, 729) @ (729, 27) = (228146,27)

    counts = logits.exp() 
    probs = counts/counts.sum(1, keepdim=True) 
    loss = -probs[torch.arange(num), ys].log().mean()
    print(loss.item())
    
    #-----------Backward pass--------------
    W.grad = None # set weights gradients to zero
    loss.backward()

    #------------Update weights--------------
    W.data += -70 * W.grad

3.7901272773742676
3.6110477447509766
3.5029518604278564
3.42195725440979
3.353463888168335
3.293139934539795
3.2393369674682617
3.1913976669311523
3.1487584114074707
3.110745668411255
3.076633930206299
3.045764446258545
3.0176022052764893
2.9917361736297607
2.9678499698638916
2.945695161819458
2.925065517425537
2.905789375305176
2.887718439102173
2.8707275390625
2.854706048965454
2.83955979347229
2.8252079486846924
2.8115804195404053
2.798614740371704
2.78625750541687
2.7744622230529785
2.76318621635437
2.7523932456970215
2.7420494556427
2.7321255207061768
2.7225940227508545
2.713430881500244
2.704613447189331
2.6961207389831543
2.687934398651123
2.6800377368927
2.6724133491516113
2.6650469303131104
2.6579248905181885
2.651034355163574
2.6443629264831543
2.6379001140594482
2.631635904312134
2.6255595684051514
2.6196634769439697
2.613938331604004
2.608376979827881
2.602971315383911
2.597715377807617


**Note: Importance of `keepdim=True`**

`logits.sum(1, keepdim=True).shape` → `torch.Size([4, 1])`  

If we forget `keepdim=True`, PyTorch removes the summed dimension, giving a tensor of shape `[4]`.

During broadcasting with `logits` of shape `[4, 27]`, the `[4]` tensor is aligned with the **last dimension**, so it is treated as `[1, 4]`, not `[4, 1]`.

Thus, `[4, 27] / [1, 4]` is not compatible for broadcasting, whereas `[4, 27] / [4, 1]` broadcasts correctly.

Therefore, `keepdim=True` is necessary to preserve the dimension as `[4, 1]` and broadcast the sum across the 27 classes for each example.

In [91]:
# Generating names
for i in range(10):
    out = [] # for storing the output
    context = ['.', '.'] # for triggering the model to start outputting characters to form a name
    
    while True:
        pair = context[0] + context[1]
        ix = ctoi[pair]
    
        xenc = F.one_hot(torch.tensor([ix]), num_classes=729).float() 
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        # Randomly draw one index from the 27 indices, using the values in p as the sampling probabilities.
        # idx with highest probability has higher chances, but its not guaranteed
        next_ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    
        next_char = itos[next_ix]

        if next_char == '.': 
            break         # breaking when it hits end character
        out.append(next_char)
        context = [context[1], next_char]
    
    if len(out)>1:
        print(''.join(out)) 

milaver
maren
taszfrecpmkjihgra
jagyrydvyjon
jant
hjdtgvhgmhodalia
slgraiki
kizljnmvnvccabrboakluin
an
stone


## Evaluation

In [92]:
nlls = torch.zeros(5) #nll = negative log likelihood, tells quality of output, the lower the better
for i in range(5):
  # i-th trigram:
  x = xs[i].item() # input character index
  y = ys[i].item() # label character index
  print('--------')
  print(f'trigram example {i+1}: {itoc[x]}{itos[y]} (indexes {x},{y})')
  print('input to the neural net:', x)
  print('output probabilities from the neural net:', probs[i])
  print('label (actual next character):', y)
  p = probs[i, y]
  print('probability assigned by the net to the the correct character:', p.item())
  logp = torch.log(p)
  print('log likelihood:', logp.item())
  nll = -logp
  print('negative log likelihood:', nll.item())
  nlls[i] = nll

print('=========')
print('average negative log likelihood, i.e. loss =', nlls.mean().item())

--------
trigram example 1: ..e (indexes 0,5)
input to the neural net: 0
output probabilities from the neural net: tensor([0.0015, 0.1375, 0.0406, 0.0480, 0.0526, 0.0477, 0.0128, 0.0207, 0.0271,
        0.0183, 0.0755, 0.0924, 0.0489, 0.0791, 0.0356, 0.0121, 0.0159, 0.0034,
        0.0510, 0.0640, 0.0407, 0.0032, 0.0116, 0.0095, 0.0045, 0.0166, 0.0289],
       grad_fn=<SelectBackward0>)
label (actual next character): 5
probability assigned by the net to the the correct character: 0.04766594246029854
log likelihood: -3.0435380935668945
negative log likelihood: 3.0435380935668945
--------
trigram example 2: .em (indexes 5,13)
input to the neural net: 5
output probabilities from the neural net: tensor([0.0229, 0.0268, 0.0055, 0.0263, 0.0297, 0.0123, 0.0093, 0.0076, 0.0279,
        0.0528, 0.0237, 0.0018, 0.3137, 0.1743, 0.0219, 0.0048, 0.0022, 0.0036,
        0.0284, 0.0365, 0.0157, 0.0424, 0.0476, 0.0063, 0.0136, 0.0199, 0.0222],
       grad_fn=<SelectBackward0>)
label (actual next chara